# Novas premissas
- Verificou-se que o trabalho anterior estava trazendo alguns ganhos no contexto acadêmico, mas não estava fazendo sentido para atender a necessidade do cliente.
- Procurar um equilíbrio entre demandas de contexto de negócios e contexto acadêmico.
  - Contexto de negócios: Resolver um problema real, trazer algo que seja útil e de valor. Solução que reflita a realidade.
  - Contexto acadêmico: Aplicar conceitos entre as disciplinas, ter um cenário rico, fazer abstrações e generalizações.
- Logo, da busca deste equilíbrio, aguns objetivos e dados serão mantidos tal como são para se replicar a realidade, outros terão que ser ajustados ou até criados para melhor se aplicar conceitos.
- Problema primordial para se resolver: classificação de risco em ESG de empresas em: alto, médio ou baixo, usando algoritmo de ML para isto.
  - O cliente não possui uma base de dados viável para poder treinar modelos de ML
  - Alternativa é procurar uma base de dados com propósito semelhante, adaptar ele para o problema do cliente, treinar o modelo e usar este aprendizado para classificar novas empresas
  - Será utilizada o dataset do kaggle...
    - A classificação do risco, será feita a partir da pontuação total ESG
      - pontuação total vem da soma das pontuações das dimensões isoladas de E, S e G
        - Pontuações das dimensões isoladas são segmentadas a partir do nível (level) em intervalos de 200 pontos, ou a partir do grau (grade) em intervalos de 100 pontos (mais precisa). Porém, a pontuação final exata depende de um "fator misterioso", que não está claro
- É a capacidade de poder prever as pontuações das dimensões isoladas que o trabalho de machine learning terá utilidade, de forma a tentar descobrir que "fator misterioso" seria este.
  - Iremos supor que este "fator misterioso" possa ter alguma ligação com o setor (industry), bolsa (exchange) ou algum dado externo, como por exemplo faturamento anual ou quantidade de funcionários.
  - Será portanto, incluído um novo dataset com faturamento anual e tamanho, consolidados do ano de 2021 obtidos a partir do SEC 10-K / Annual Report 2021, usando o cik como chave de busca
- Será feita uma ponte entre a forma que o cliente está coletando os seus dados e o formato da base de dados que o modelo usa. Em outras plavras, iremos criar uma "tradução" entre planilhas e restrições na forma de preenchimento e respostas permitidas

## Revisão da camada silver

 A transformação ocorrerá via scrip `src/silver_transform.py`, que foi atualizado.

>A etapa Silver é uma etapa de transformação de dados com o objetivo de se normalizar ou corrigir problemas detectados na EDA e também eliminar dados desnecessários, para deixar o arquivo mais leve para armazenamento e processamento.

### Pipeline de Transformação de Dados

Abaixo, todas as transformações que irão ocorrer para a camada Silver:


1. Carrega o dado bruto de `data/bronze/data.csv`

   ↓

2. Descarta colunas sem utilidade  
   - `logo`  
   - `weburl`  
   - `currency`  

   ↓

3. Reordena colunas 
   - `cik`
   - `ticker`
   - `name`
   - `exchange`
   - `industry`
   - `environment_grade`
   - `environment_level`
   - `environment_score`
   - `social_grade`
   - `social_level`
   - `social_score`
   - `governance_grade`
   - `governance_level`
   - `governance_score`
   - `total_grade`
   - `total_level`
   - `total_score`
   - `last_processing_date`

   ↓

4. Corrige inconsistências em `industry`  
   - Padroniza grafias diferentes (and, & e " ")

   ↓

5. Preenche valores nulos em `industry`  
   - Busca informações via `yfinance` usando o ticker  
   - Fallback para `"Unknown"`

   ↓

6. Converte `last_processing_date` para `datetime`

   ↓

7. Padroniza a coluna `cik` para o padrão da SEC (U.S. Securities and Exchange Commission)
   - zero-padding de 10 dígitos preenchidos com zeros à esquerda

   ↓

8. Padroniza a coluna `ticker` para maiúsculas, que é o formato padrão

   ↓

9. Simplifica a coluna `exchange` para siglas, convertendo "NEW YORK STOCK EXCHANGE, INC." para "NYSE" e "NASDAQ NMS - GLOBAL MARKET" para "NASDAQ"

   ↓

10. Inclui as colunas `revenue_M` e `employees` a partir respectivamente das colunas `revenue_2021_millions_usd` e `employees_2021` do dataset `data/bronze\company_size_revenue_2021.csv`, usando o `cik` como chave de junção. Estas colunas serão incluídas logo após a coluna `industry`, da esquerda para a direita.

   ↓

11. Salva o resultado em `data/silver/data_silver.csv`

   ↓

12. Registra `metadata_silver.json`